# Notebook 01 — Dataset Preparation

Prepares user-downloaded datasets for ZimAgriTrust training. This notebook does not train models. It validates and normalizes dataset layout for later notebooks.

In [1]:
import os, json, shutil, hashlib, time
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image

DESKTOP_DATA_ROOT = Path(r'C:\Users\MJ\Desktop')
CROP_DISEASE_DIR = Path(os.getenv('CROP_DISEASE_DIR', r'C:\Users\MJ\Desktop\CROP DESEASE'))
CROP_IMAGES_DIR = Path(os.getenv('CROP_IMAGES_DIR', r'C:\Users\MJ\Desktop\crop images'))
CROP_DATASETS_DIR = Path(os.getenv('CROP_DATASETS_DIR', r'C:\Users\MJ\Desktop\CROP DATA SETS'))
CROPS_DIR = Path(os.getenv('CROPS_DIR', r'C:\Users\MJ\Desktop\crops'))
PROCESSED_DIR = Path(os.getenv('PROCESSED_DATA_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\data'))
IMG_SIZE = int(os.getenv('PREP_IMG_SIZE', '224'))
SPLIT_SEED = int(os.getenv('SPLIT_SEED', '42'))

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('CROP_DISEASE_DIR:', CROP_DISEASE_DIR)
print('CROP_IMAGES_DIR:', CROP_IMAGES_DIR)
print('CROP_DATASETS_DIR:', CROP_DATASETS_DIR)
print('CROPS_DIR:', CROPS_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

CROP_DISEASE_DIR: C:\Users\MJ\Desktop\CROP DESEASE
CROP_IMAGES_DIR: C:\Users\MJ\Desktop\crop images
CROP_DATASETS_DIR: C:\Users\MJ\Desktop\CROP DATA SETS
CROPS_DIR: C:\Users\MJ\Desktop\crops
PROCESSED_DIR: C:\Users\MJ\Desktop\Agric\jupyter\data


In [2]:
CROP_CLASSES = ['maize', 'mango', 'tomato', 'soya_beans', 'groundnuts', 'tobacco', 'cotton', 'cabbage', 'potato', 'onion', 'sugar_beans', 'sunflower']
DISEASE_CLASSES = ['healthy', 'rust', 'blight', 'powdery_mildew', 'mosaic_virus', 'leaf_spot', 'rot']
QUALITY_CLASSES = ['A', 'B', 'C']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
MIN_WIDTH = int(os.getenv('MIN_IMAGE_WIDTH', '96'))
MIN_HEIGHT = int(os.getenv('MIN_IMAGE_HEIGHT', '96'))
MAX_IMAGES_PER_CLASS = int(os.getenv('MAX_IMAGES_PER_CLASS', '0'))

CROP_KEYWORDS = {
    'maize': ['maize', 'corn'],
    'mango': ['mango'],
    'tomato': ['tomato'],
    'soya_beans': ['soya', 'soybean', 'soybeans', 'soy'],
    'groundnuts': ['groundnut', 'groundnuts', 'peanut', 'peanuts'],
    'tobacco': ['tobacco'],
    'cotton': ['cotton'],
    'cabbage': ['cabbage'],
    'potato': ['potato'],
    'onion': ['onion'],
    'sugar_beans': ['sugar bean', 'sugar_beans', 'sugarbeans', 'bean', 'beans'],
    'sunflower': ['sunflower'],
}
DISEASE_KEYWORDS = {
    'healthy': ['healthy', 'normal'],
    'rust': ['rust'],
    'blight': ['blight'],
    'powdery_mildew': ['powdery', 'mildew'],
    'mosaic_virus': ['mosaic', 'virus'],
    'leaf_spot': ['leaf spot', 'leaf_spot', 'spot'],
    'rot': ['rot'],
}
QUALITY_KEYWORDS = {
    'A': ['grade a', 'grade_a', 'quality a', 'class a', '\\a\\', '/a/'],
    'B': ['grade b', 'grade_b', 'quality b', 'class b', '\\b\\', '/b/'],
    'C': ['grade c', 'grade_c', 'quality c', 'class c', '\\c\\', '/c/'],
}

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def image_fingerprint(img: Image.Image) -> str:
    small = img.convert('RGB').resize((16, 16))
    return hashlib.sha1(small.tobytes()).hexdigest()

def infer_label(path: Path, keyword_map):
    text = str(path).lower().replace('-', ' ').replace('_', ' ')
    for label, keywords in keyword_map.items():
        if any(keyword.lower() in text for keyword in keywords):
            return label
    return None

def collect_labeled_images(source_roots, keyword_map):
    labeled = {label: [] for label in keyword_map}
    skipped = {'missing_roots': [], 'unsupported_ext': 0, 'unlabeled': 0, 'corrupt': 0, 'too_small': 0, 'duplicates': 0}
    seen = set()
    for root in source_roots:
        if not root.exists():
            skipped['missing_roots'].append(str(root))
            continue
        for path in root.rglob('*'):
            if not path.is_file():
                continue
            if path.suffix.lower() not in IMAGE_EXTS:
                skipped['unsupported_ext'] += 1
                continue
            label = infer_label(path, keyword_map)
            if label is None:
                skipped['unlabeled'] += 1
                continue
            try:
                with Image.open(path) as img:
                    img.verify()
                with Image.open(path) as img:
                    img = img.convert('RGB')
                    if img.width < MIN_WIDTH or img.height < MIN_HEIGHT:
                        skipped['too_small'] += 1
                        continue
                    fp = image_fingerprint(img)
            except Exception:
                skipped['corrupt'] += 1
                continue
            if fp in seen:
                skipped['duplicates'] += 1
                continue
            seen.add(fp)
            labeled[label].append(path)
    return labeled, skipped

def normalize_image(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(src) as img:
        img = img.convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        img.save(dst.with_suffix('.jpg'), quality=92)

In [3]:
rng = np.random.default_rng(SPLIT_SEED)
summary = {'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'image_size': IMG_SIZE, 'datasets': {}}

def prepare_classification_dataset(source_roots, classes, output_name, keyword_map):
    output_root = PROCESSED_DIR / output_name
    labeled_images, skipped = collect_labeled_images(source_roots, keyword_map)
    counts = {split: {cls: 0 for cls in classes} for split in ['train', 'val', 'test']}
    raw_counts = {cls: len(labeled_images.get(cls, [])) for cls in classes}

    for cls in classes:
        images = list(labeled_images.get(cls, []))
        if MAX_IMAGES_PER_CLASS > 0:
            images = images[:MAX_IMAGES_PER_CLASS]
        if not images:
            continue
        rng.shuffle(images)
        n = len(images)
        train_end = max(1, int(n * 0.70)) if n >= 3 else n
        val_end = int(n * 0.85) if n >= 7 else train_end
        splits = {'train': images[:train_end], 'val': images[train_end:val_end], 'test': images[val_end:]}
        for split, split_images in splits.items():
            for idx, src in enumerate(split_images):
                safe_name = ''.join(ch if ch.isalnum() else '_' for ch in src.stem)[:80]
                dst = output_root / split / cls / f'{safe_name}_{idx}.jpg'
                try:
                    normalize_image(src, dst)
                    counts[split][cls] += 1
                except Exception:
                    skipped['corrupt'] += 1

    summary['datasets'][output_name] = {'raw_labeled_counts': raw_counts, 'processed_counts': counts, 'skipped': skipped}
    print('\n' + output_name.upper())
    print('Raw labeled counts:', raw_counts)
    print('Processed counts:', counts)
    print('Skipped:', skipped)

prepare_classification_dataset([CROP_IMAGES_DIR, CROPS_DIR], CROP_CLASSES, 'crops', CROP_KEYWORDS)
prepare_classification_dataset([CROP_DISEASE_DIR], DISEASE_CLASSES, 'diseases', DISEASE_KEYWORDS)
prepare_classification_dataset([CROP_IMAGES_DIR, CROPS_DIR], QUALITY_CLASSES, 'quality', QUALITY_KEYWORDS)

C:\Users\MJ\anaconda3\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



CROPS
Raw labeled counts: {'maize': 304, 'mango': 76, 'tomato': 90, 'soya_beans': 94, 'groundnuts': 0, 'tobacco': 0, 'cotton': 0, 'cabbage': 82, 'potato': 141, 'onion': 80, 'sugar_beans': 0, 'sunflower': 0}
Processed counts: {'train': {'maize': 212, 'mango': 53, 'tomato': 62, 'soya_beans': 65, 'groundnuts': 0, 'tobacco': 0, 'cotton': 0, 'cabbage': 57, 'potato': 98, 'onion': 56, 'sugar_beans': 0, 'sunflower': 0}, 'val': {'maize': 46, 'mango': 11, 'tomato': 14, 'soya_beans': 14, 'groundnuts': 0, 'tobacco': 0, 'cotton': 0, 'cabbage': 12, 'potato': 21, 'onion': 12, 'sugar_beans': 0, 'sunflower': 0}, 'test': {'maize': 46, 'mango': 12, 'tomato': 14, 'soya_beans': 15, 'groundnuts': 0, 'tobacco': 0, 'cotton': 0, 'cabbage': 13, 'potato': 22, 'onion': 12, 'sugar_beans': 0, 'sunflower': 0}}
Skipped: {'missing_roots': [], 'unsupported_ext': 7, 'unlabeled': 3743, 'corrupt': 0, 'too_small': 0, 'duplicates': 315}

DISEASES
Raw labeled counts: {'healthy': 5732, 'rust': 3410, 'blight': 6188, 'powdery_

In [4]:
def clean_csv(src_name, dst_folder, required_columns):
    src = CROP_DATASETS_DIR / src_name
    dst_dir = PROCESSED_DIR / dst_folder
    dst_dir.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        print(f'Missing optional CSV: {src}')
        return None
    df = pd.read_csv(src)
    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(f'{src_name} missing columns: {missing}')
    df = df.drop_duplicates().dropna(subset=required_columns)
    dst = dst_dir / src_name
    df.to_csv(dst, index=False)
    summary['datasets'][src_name] = {'rows': int(len(df)), 'path': str(dst)}
    print(f'Cleaned {src_name}: {len(df)} rows -> {dst}')
    return dst

clean_csv('price_history.csv', 'prices', ['date', 'crop_type', 'quantity_kg', 'price_usd'])
clean_csv('synthetic_transactions.csv', 'transactions', ['amount', 'quantity', 'price_per_kg', 'hour', 'farmer_history', 'buyer_history'])
clean_csv('weather_history.csv', 'weather', ['date'])
clean_csv('soil_data.csv', 'soil', [])

Missing optional CSV: C:\Users\MJ\Desktop\CROP DATA SETS\price_history.csv
Missing optional CSV: C:\Users\MJ\Desktop\CROP DATA SETS\synthetic_transactions.csv
Missing optional CSV: C:\Users\MJ\Desktop\CROP DATA SETS\weather_history.csv
Missing optional CSV: C:\Users\MJ\Desktop\CROP DATA SETS\soil_data.csv


In [5]:
manifest_path = PROCESSED_DIR / 'dataset_manifest.json'
manifest_path.write_text(json.dumps(summary, indent=2))
print('Dataset preparation complete:', manifest_path)
print(json.dumps(summary, indent=2))

Dataset preparation complete: C:\Users\MJ\Desktop\Agric\jupyter\data\dataset_manifest.json
{
  "created_at": "2026-05-11T20:44:24Z",
  "image_size": 224,
  "datasets": {
    "crops": {
      "raw_labeled_counts": {
        "maize": 304,
        "mango": 76,
        "tomato": 90,
        "soya_beans": 94,
        "groundnuts": 0,
        "tobacco": 0,
        "cotton": 0,
        "cabbage": 82,
        "potato": 141,
        "onion": 80,
        "sugar_beans": 0,
        "sunflower": 0
      },
      "processed_counts": {
        "train": {
          "maize": 212,
          "mango": 53,
          "tomato": 62,
          "soya_beans": 65,
          "groundnuts": 0,
          "tobacco": 0,
          "cotton": 0,
          "cabbage": 57,
          "potato": 98,
          "onion": 56,
          "sugar_beans": 0,
          "sunflower": 0
        },
        "val": {
          "maize": 46,
          "mango": 11,
          "tomato": 14,
          "soya_beans": 14,
          "groundnuts": 0,
   